## 1. ariketa

In [64]:
def load(filename, encoding='utf8'):
    with open(filename, encoding=encoding) as f:
        h = {}
        for line in f :
            timestamp,user,action,resource = line.split()
            h.setdefault(user,[]).append((timestamp,action,resource))
        return h

In [65]:
def save(logs, filename, encoding='utf8'):
    with open(filename, mode='w', encoding=encoding) as f:
        for user,z in logs.items():
            for timestamp,action,resource in z :
                print(timestamp,user,action,resource,file=f)   

In [67]:
def check_in(value, group, error_msg):
    if value not in group :
        raise ValueError(error_msg)
        
def resources(logs, user):
    check_in(user, logs, f'Unknown user: {user}')
    return {resource for _,_,resource in logs[user] if resource!= "-"}            

In [68]:
def actions(logs, user):
    check_in(user, logs, f'Unknown user: {user}')
    h = dict.fromkeys(['login','logout','write','read'], 0)
    for _,action,_ in logs[user]:
        h[action] += 1
    return h

In [69]:
def most_frequent_resources(logs, action_type):
    check_in(action_type, ['login','logout','write','read'], f'Unknown action type: {action_type}')
    h = {}
    for user,z in logs.items():
        for _,action,resource in z :
            if action == action_type and resource != "-" :
                h[resource] = h.get(resource,0) + 1
    max_count = max(h.values(), default=0)
    return [resource for resource,count in h.items() if count == max_count]

## 2. ariketa

Problemaren tamaina, `v` zerrendaren elementu kopurua, `n=len(v)`

In [105]:
def missing_v1(v):               #  kasu ona  <-->  kasu txarra
    z = []                       # 1
    for i in range(1,len(v)+1):  # n x 1
        if i not in v:           # 1+2+...+n  <--> 1+(n-1)xn
            z.append(i)          # 0          <--> n-1
    return z                     # 1

def missing_v1b(v): # aurrekoaren baliokidea
    return [i for i in range(1,len(v)+1) if i not in v] 

Kasu on edo txarrik egotekotan, ondorengoak izan litezke:
* **Kasu ona**: bektorean [1,n] tarteko zenbaki guztiak egotea &rarr; $t(n) = \frac{n \cdot (n+1)}{2} + n + 2$
* **Kasu txarra**: bektorean [1,n] tarteko zenbaki bakarra egotea &rarr; $t(n) = n^2 + n + 2$

Beraz, funtzioak orden zehatza du: $\Theta(n^2)$


**OHARRA**: Egiazki, problema honek soluzio lineal bat du, ordenazio, multzo edo hiztegirik erabili gabe. Izan ere, bilatzen ari garen balioak $[1,n]$ arteko zenbaki osoak dira eta beraz zerrenden indize gisa erabili daitezke, hiztegi bat balitz bezala:

In [84]:
def missing_v1b(v):
    agertu = [False] * (len(v)+1)  # agertu[0] ez dugu erabiliko
    for i in v :
        agertu[i] = True # v-n agertzen diren elementuak "markatu"
    z = []
    for i in range(1, len(v)+1): # v-n ez dauden [1,n] tarteko elementuak z-n gorde
        if not agertu[i]:
            z.append(i)
    return z

In [104]:
def missing_v1c(v): # Aurrekoaren baliokidea
    agertu = list(range(len(v)+1))
    for i in v :
        agertu[i] = 0
    return [i for i in agertu if i]  # 0-ren ezberdinak diren balio guztiak

In [103]:
def missing_v1d(v): # Aurrekoaren baliokidea
    agertu = list(range(len(v)+1))
    for i in v :
        agertu[i] = 0
    return list(filter(None,agertu))

In [62]:
def missing_v2(v):                      #  kasu ona  <-->  kasu txarra
    w = [0] + sorted(v) + [len(v)+1]    # nxlogn + 2 + (n+2)
    z = []                              # 1
    for i in range(len(w)-1):           # (n+1) x 1
        z.extend(range(w[i]+1,w[i+1]))  # n+1        <--> n-1 + n-1
    return z                            # 1

def missing_v2b(v): # aurrekoaren baliokidea
    w = [0] + sorted(v) + [len(v)+1]
    z = []
    for i,j in zip(w,w[1:]):
        z.extend(range(i+1,j))
    return z

def missing_v2c(v): # aurrekoaren baliokidea
    w = [0] + sorted(v) + [len(v)+1]
    return [k for i,j in zip(w,w[1:]) for k in range(i+1,j)]

Kasu on edo txarrik egotekotan, ondorengoak izan litezke:
* **Kasu ona**: bektorean [1,n] tarteko zenbaki guztiak egotea &rarr; $t(n) = n \cdot \log n + 3n + 8$
* **Kasu txarra**: bektorean [1,n] tarteko zenbaki bakarra egotea &rarr; $t(n) = n \cdot \log n + 4n + 5$

Beraz, funtzioak orden zehatza du: $\Theta(n \cdot \log n)$


In [79]:
def missing_v3(v):
    s = set(v)                  # n
    z = []                      # 1
    for i in range(1,len(v)+1): # n x 1
        if i not in s:          # | n x 1
            z.append(i)         # |
    return z                    # 1

def missing_v3b(v): # aurrekoaren baliokidea
    s = set(v)
    return [i for i in range(1,len(v)+1) if i not in s]

$t(n) = 3n + 2$ &rarr;  $\Theta(n)$

## 3. ariketa

In [133]:
class SparseVector(object):
    
    def __init__(self, length, it=()):
        self.length = length
        self.v = {}
        for index,value in it:
            self[index] = value
    
    def check_index(self, index):
        if type(index) != int or index < 0 or index >= self.length :
            raise IndexError(f'wrong index: {index}')

    def check_value(self, value):
        if type(value) not in (int,float) :
            raise ValueError(f'wrong value type: {type(value)}')
            
    def __setitem__(self, index, value):
        self.check_index(index)
        self.check_value(value)
        self.v[index] = value
        if value == 0 :
            del self.v[index]
            
    def __getitem__(self, index):
        self.check_index(index)
        return self.v.get(index, 0)
        
    def __contains__(self, index):
        self.check_index(index)
        return index in self.v
    
    def __len__(self):
        return self.length
    
    def rank(self):
        return len(self.v)
    
    def __eq__(self, other):
        return type(other)==SparseVector and len(self)==len(other) and self.v==other.v
    
    def __iter__(self):
        return iter(self.v)
    
    def __add__(self, other):
        if type(other)!=SparseVector or len(self)!=len(other) :
            raise ValueError(f'cannot add')
        sv = SparseVector(len(self))
        for i in self :
            sv[i] = sv[i] + self[i]
        for i in other :
            sv[i] = sv[i] + other[i]
        return sv
    
    def prod(self, factor):
        self.check_value(factor)
        sv = SparseVector(len(self))
        if factor != 0 :
            
        
        
v = SparseVector(1000,[(3,120),(23,-100)])
print(*(v[i] for i in range(10)))
v[7]= -1
print(*(v[i] for i in range(10)))
print(*(i in v for i in range(10)))
print(len(v), v.rank())
print(v==[1,2,3])
print(v==SparseVector(1000,[(3,120),(7,-1),(23,-100)]))
print(v==SparseVector(50,  [(3,120),(7,-1),(23,-100)]))
print(*v)
v2 = SparseVector(1000,[(3,120),(23,100)])
v3 = v + v2
print(v.v, v2.v, v3.v)


0 0 0 120 0 0 0 0 0 0
0 0 0 120 0 0 0 -1 0 0
False False False True False False False True False False
1000 3
False
True
False
3 23 7
{3: 120, 23: -100, 7: -1} {3: 120, 23: 100} {3: 240, 7: -1}
